# Issue 001 — Colab Environment & Dataset Manifest

This notebook stands up the **reproducible Google Colab foundation** shared by every pipeline
(A — raw-video VideoMAE, B — skeleton temporal, C — hybrid fusion). It does **not** download
datasets or train models — those belong to later issues. Its only jobs are:

1. Mount Google Drive and create the persistent project directory layout.
2. Install the **approved, Colab-compatible** dependency stack.
   *Never* reinstalls `torch` or `torchvision` (Colab ships a CUDA-matched build; reinstalling
   risks pulling a CPU-only wheel and breaking every GPU model).
3. Capture a literal `pip freeze` as the environment lock file.
4. Capture GPU / CUDA / Python / timestamp + resolved Drive paths into a JSON run log.

Re-running this notebook is safe and idempotent.

---

## 1. Mount Google Drive

Mounting is the only side-effect that requires user interaction (`Allow` on the OAuth prompt).
If you are not running on Colab (e.g. running locally for smoke-testing this notebook), the
mount call will raise — that's expected, and `FALL_DETECTION_DRIVE_ROOT` can be pointed at any
writable path instead.

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    print("google.colab not available — running outside Colab. Set FALL_DETECTION_DRIVE_ROOT "
          "to a writable path if you want setup to write to a Drive-like location.")
    IN_COLAB = False

## 2. Put `colab/` on `sys.path` so `import setup` works

We clone or copy the repo into the runtime, then add the project root to `sys.path`. The
setup helper lives at `colab/setup.py` and exposes a single `run_setup()` entry point.

In [ ]:
import os
import pathlib
import sys

# Adjust if your repo lives elsewhere on Drive.
PROJECT_ROOT = pathlib.Path(os.environ.get("FALL_DETECTION_PROJECT_ROOT", "/content/fall_detection"))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from colab import setup  # noqa: E402
print("setup module loaded from:", setup.__file__)

## 3. Install the approved dependency stack

`setup.install_approved_packages()` runs **one** `pip install` with the exact approved list.
It does **not** touch `torch` or `torchvision`.

TrackEval is intentionally excluded — it has no clean pip wheel. Install it manually only when
MOT metrics from TrackEval are actually needed (see `setup.TRACKEVAL_INSTALL_NOTE`).

In [ ]:
setup.install_approved_packages(quiet=True)
print("Approved packages installed:")
for pkg in setup.APPROVED_PIP_PACKAGES:
    print("  -", pkg)

## 4. Verify Colab's existing torch is intact

Sanity check — if torch shows up as CPU-only after the install, something has silently broken.
This is a cheap, fast check that catches the most common Colab pitfall.

In [ ]:
import torch  # noqa: E402

print(f"torch version     : {torch.__version__}")
print(f"torch CUDA version: {torch.version.cuda}")
print(f"CUDA available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device count      : {torch.cuda.device_count()}")
    print(f"Device 0          : {torch.cuda.get_device_name(0)}")
assert torch.cuda.is_available(), "CUDA not available — check Colab runtime type."
assert torch.backends.cudnn.is_available(), "cuDNN not available — torch is probably broken."

## 5. Create the persistent Drive layout

Directories created on Drive (under `MyDrive/fall_detection/` by default; override via the
`FALL_DETECTION_DRIVE_ROOT` environment variable):

| Path                  | Purpose                                   |
|-----------------------|-------------------------------------------|
| `datasets/`           | Raw + staged datasets (read-only cache)   |
| `artifacts/`          | Precomputed crops, skeletons, env lock    |
| `checkpoints/`        | Training checkpoints (resumable)          |
| `metrics/`            | Eval results, golden-set scores           |
| `logs/`               | Per-run logs, run metadata JSON           |

All are created via `DriveLayout.ensure()` and are safe to re-run.

In [ ]:
layout = setup.DriveLayout.resolve()
layout.ensure()
print("Drive layout:")
for name in setup.DRIVE_SUBDIRS:
    print(f"  {name:<12} -> {getattr(layout, name)}")

## 6. Capture the environment lock + run log

Two artifacts are written to Drive:

- `artifacts/pip_freeze.txt` — a **literal** `pip freeze`, the project's environment lock.
  It includes everything in the runtime (Colab base + approved installs) so any drift in
  Colab's base image is reflected in version control.
- `logs/setup_run_log.json` — JSON snapshot of GPU name, VRAM, CUDA version, Python version,
  UTC timestamp, and the resolved Drive paths. Tag with the issue number so every run is
  traceable.

In [ ]:
artifacts = setup.run_setup(
    install_deps=False,        # already done in step 3; skip here for idempotency
    write_freeze=True,
    write_run_log=True,
    extra_log_fields={"issue": "001", "notebook": "colab/001_setup.ipynb"},
)
for name, path in artifacts.items():
    print(f"{name:<10} -> {path}")

## 7. Done

Setup is complete. Subsequent issues (002 — YOLO+ByteTrack front-end, 003 — clip generation,
006 — Pipeline A training, etc.) can now assume:

- The approved dependency stack is installed and verified.
- The Drive layout exists and is writable.
- An environment lock and a run log are committed under `artifacts/` and `logs/`.

Re-run this notebook any time the runtime resets or when you want to refresh the env lock.